In [1]:
# ============================================
# CELDA 4 — CORRER EL VIERNES / SÁBADO
# Imports básicos
# ============================================

import pickle
import pandas as pd
import numpy as np
import json

In [2]:
# ============================================
# CELDA 5 — CORRER EL VIERNES / SÁBADO
# Cargar modelo desde disco
# ============================================

with open('modelo_premier.pkl', 'rb') as f:
    datos = pickle.load(f)

model_v3        = datos['model_v3']
feature_cols_v3 = datos['feature_cols_v3']
le              = datos['le']
df              = datos['df']

print("Modelo cargado correctamente")

Modelo cargado correctamente


In [3]:
# ============================================
# CELDA 6 — CORRER EL VIERNES / SÁBADO
# Cargar funciones de seguimiento
# ============================================

predicciones = []

def registrar_prediccion(jornada, fecha, home, away, pred, prob_H, prob_D, prob_A, confianza):
    predicciones.append({
        'jornada': jornada, 'fecha': fecha,
        'home': home, 'away': away,
        'prediccion': pred, 'prob_H': prob_H,
        'prob_D': prob_D, 'prob_A': prob_A,
        'confianza': confianza, 'resultado': None, 'correcto': None
    })
    print(f"Registrado: {home} vs {away} → {pred} ({confianza:.1%})")

def registrar_resultado(home, away, resultado_real):
    for p in predicciones:
        if p['home'] == home and p['away'] == away:
            p['resultado'] = resultado_real
            p['correcto']  = p['prediccion'] == resultado_real
            estado = "ACERTADO" if p['correcto'] else "FALLADO"
            print(f"{estado}: {home} vs {away} | Pred: {p['prediccion']} | Real: {resultado_real}")
            return
    print(f"Partido no encontrado: {home} vs {away}")

def ver_seguimiento():
    if not predicciones:
        print("Sin predicciones registradas aún")
        return
    df_seg    = pd.DataFrame(predicciones)
    total     = len(df_seg)
    jugados   = df_seg['resultado'].notna().sum()
    acertados = df_seg['correcto'].sum() if jugados > 0 else 0
    accuracy  = acertados / jugados if jugados > 0 else 0
    print(f"\n{'='*50}")
    print(f"  RESUMEN GENERAL")
    print(f"{'='*50}")
    print(f"  Total predicciones:  {total}")
    print(f"  Partidos jugados:    {jugados}")
    print(f"  Acertados:           {int(acertados)}")
    print(f"  Accuracy real:       {accuracy:.1%}")
    print(f"  Baseline:            41.2%")
    print(f"  Diferencia:          {(accuracy - 0.412):+.1%}")
    print(f"\n{'='*50}")
    print(f"  DETALLE POR PARTIDO")
    print(f"{'='*50}")
    for _, row in df_seg.iterrows():
        if row['resultado'] is not None:
            icon = "✓" if row['correcto'] else "✗"
            print(f"  {icon} J{row['jornada']} {row['home']} vs {row['away']}")
            print(f"    Pred: {row['prediccion']} ({row['confianza']:.1%}) | Real: {row['resultado']}")
        else:
            print(f"  ? J{row['jornada']} {row['home']} vs {row['away']}")
            print(f"    Pred: {row['prediccion']} ({row['confianza']:.1%}) | Pendiente")

def guardar_predicciones(archivo='predicciones.json'):
    datos = []
    for p in predicciones:
        p_copy = p.copy()
        p_copy['correcto'] = bool(p['correcto']) if p['correcto'] is not None else None
        datos.append(p_copy)
    with open(archivo, 'w') as f:
        json.dump(datos, f, indent=2)
    print(f"Guardado: {len(datos)} predicciones")

def cargar_predicciones(archivo='predicciones.json'):
    global predicciones
    try:
        with open(archivo, 'r') as f:
            predicciones = json.load(f)
        print(f"Cargado: {len(predicciones)} predicciones")
    except FileNotFoundError:
        print("No hay predicciones guardadas aún")

print("Funciones cargadas correctamente")

Funciones cargadas correctamente


In [4]:
# ============================================
# CELDA 7 — CORRER EL VIERNES / SÁBADO
# Cargar predicciones guardadas
# ============================================

cargar_predicciones()
ver_seguimiento()

Cargado: 2 predicciones

  RESUMEN GENERAL
  Total predicciones:  2
  Partidos jugados:    0
  Acertados:           0
  Accuracy real:       0.0%
  Baseline:            41.2%
  Diferencia:          -41.2%

  DETALLE POR PARTIDO
  ? J32 West Ham vs Wolves
    Pred: H (55.2%) | Pendiente
  ? J32 Arsenal vs Bournemouth
    Pred: H (64.8%) | Pendiente


In [5]:
# # ============================================
# # CELDA 8 — CORRER DESPUÉS DE CADA PARTIDO
# # Cambiar 'H' / 'D' / 'A' por el resultado real
# #
# # H = gana el local
# # D = empate
# # A = gana el visitante
# # ============================================

# # Viernes 10 abril
# registrar_resultado('West Ham',  'Wolves',      'H')  # ← cambiar por resultado real

# # Sábado 11 abril — correr después del partido
# registrar_resultado('Arsenal',   'Bournemouth', 'H')  # ← cambiar por resultado real

In [6]:
# # ============================================
# # CELDA 9 — CORRER DESPUÉS DE CADA RESULTADO
# # Ver resumen y guardar
# # ============================================

# ver_seguimiento()
# guardar_predicciones()